# Finance MiniGPT: A Transformer Built From Scratch on FOMC Statements

**Research question.** Can a small, transparent, character-level decoder-only Transformer built from first principles learn measurable language regularities from official FOMC policy-decision statements while preserving chronological validation integrity?

This educational Week 3 capstone is a finance-language experiment, not a forecasting model, financial backtest, trading system, investment recommendation, or factual monetary-policy oracle. It makes no claim of superiority over other implementations. We build each important operation directly so its tensor shapes, causal behavior, and limitations remain inspectable.


## 1. Runtime and deterministic configuration

The candidate baseline is deliberately labeled **provisional**. Batch size, learning rate, and the canonical token budget remain subject to Phase 3B smoke-run approval. CUDA is preferred, then MPS, then CPU. Seeds and deterministic settings reduce avoidable variation, although exact floating-point equality across devices is not guaranteed. AMP is not part of the canonical path.


In [ ]:
from __future__ import annotations

import hashlib, json, math, os, platform, random, re, tempfile, time
from collections import Counter
from dataclasses import asdict, dataclass
from datetime import date
from pathlib import Path
from typing import Iterator, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

CONFIG = {
    "phase": "3A-validation-only",
    "allow_canonical_training": False,
    "seed": 0,
    "architecture": {"block_size": 128, "d_model": 256, "num_heads": 8, "num_layers": 4, "d_ff": 1024, "dropout": 0.1},
    "optimizer_candidates": {"name": "AdamW", "learning_rates": [3e-4, 5e-4], "weight_decay": 0.1, "betas": [0.9, 0.95], "grad_clip": 1.0},
    "training_candidates": {"batch_sizes": [16, 32], "token_budgets": [2_000_000, 5_000_000], "warmup_fraction": 0.05, "min_lr_fraction": 0.1, "validate_every": 250},
    "evaluation": {"validation_split": "2025", "block_sizes_reported": [64, 128, 256]},
    "generation": {"max_new_tokens": 200, "temperature": 0.8, "top_k": 20},
    "artifacts": {
        "best_checkpoint": "finance_minigpt_best.pt",
        "final_checkpoint": "finance_minigpt_final.pt",
        "metrics": "finance_minigpt_metrics.csv",
        "run_record": "finance_minigpt_run.json",
        "samples": "finance_minigpt_samples.json",
        "training_plot": "finance_minigpt_training.png",
        "attention_plot": "finance_minigpt_attention.png",
    },
    "corpus": {"id": "fomc-statements-2015-2025-v2", "sha256": "c632b6a2e7bcfc0360e9fe18113a2ec1c9edce2d42f83b4ff96f5ce4d74ea125"},
}

TEST_RESULTS = {}

def record_check(name: str, condition, detail: str = "") -> bool:
    passed = bool(condition)
    if not passed:
        suffix = f" {detail}" if detail else ""
        raise AssertionError(f"Phase 3A check failed: {name}.{suffix}")
    TEST_RESULTS[name] = passed
    return passed

def select_device() -> torch.device:
    if torch.cuda.is_available(): return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

DEVICE = select_device()
seed_everything(CONFIG["seed"])
print({"phase": CONFIG["phase"], "device": str(DEVICE), "torch": torch.__version__, "seed": CONFIG["seed"]})


## 2. Frozen FOMC corpus provenance and integrity

Only the committed UTF-8 corpus and manifest are read. The resolver supports execution from the notebook directory, a repository root, or a Colab clone by searching plausible parents for both frozen files. Parsing is lossless: exact structural blocks are validated against the manifest, while only statement bodies are returned for modeling. Any mismatch raises an actionable error; no network access is used.


In [ ]:
EXPECTED = {
    "corpus_id": "fomc-statements-2015-2025-v2",
    "sha256": "c632b6a2e7bcfc0360e9fe18113a2ec1c9edce2d42f83b4ff96f5ce4d74ea125",
    "bytes": 265669, "characters": 265669, "documents": 90,
    "train": 82, "validation": 8, "extraction_method": "federalreserve_article_paragraphs_v2",
    "normalization_version": "fomc-finance-preserving-v1",
}
START, END = "<|fomc_statement|>", "<|end_fomc_statement|>"

def resolve_corpus_dir() -> Path:
    starts = [Path.cwd()]
    if os.environ.get("WEEK03_REPO_ROOT"): starts.append(Path(os.environ["WEEK03_REPO_ROOT"]).resolve())
    if "__file__" in globals(): starts.append(Path(__file__).resolve().parent)
    for start in starts:
        for parent in [start, *start.parents]:
            for candidate in (parent, parent / "capstones" / "week03_transformers"):
                if (candidate / "fomc_statements_2015_2025.txt").is_file() and (candidate / "fomc_statements_2015_2025_manifest.json").is_file():
                    return candidate
    raise FileNotFoundError("Frozen FOMC corpus files not found from cwd/notebook parents.")

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def load_verified_corpus(corpus_dir: Path):
    corpus_path = corpus_dir / "fomc_statements_2015_2025.txt"
    manifest_path = corpus_dir / "fomc_statements_2015_2025_manifest.json"
    raw_bytes = corpus_path.read_bytes()
    text = raw_bytes.decode("utf-8")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    checks = {
        "sha256": hashlib.sha256(raw_bytes).hexdigest() == EXPECTED["sha256"] == manifest["corpus_sha256"],
        "bytes": len(raw_bytes) == EXPECTED["bytes"] == manifest["corpus_utf8_bytes"],
        "characters": len(text) == EXPECTED["characters"] == manifest["corpus_characters"],
        "corpus_id": manifest["corpus_id"] == EXPECTED["corpus_id"],
        "extraction_method": manifest["extraction_method"] == EXPECTED["extraction_method"],
        "normalization_version": manifest["normalization_version"] == EXPECTED["normalization_version"],
        "document_count": manifest["document_count"] == EXPECTED["documents"] == len(manifest["documents"]),
        "split_counts": manifest["train_document_count"] == EXPECTED["train"] and manifest["validation_document_count"] == EXPECTED["validation"],
        "zero_failures": len(manifest["failures"]) == 0,
        "marker_counts": text.count(START) == text.count(END) == EXPECTED["documents"],
    }
    failed = [name for name, ok in checks.items() if not ok]
    if failed: raise ValueError(f"Corpus integrity mismatch: {failed}")

    pattern = re.compile(
        rf"{re.escape(START)}\n"
        r"date: (?P<date>\d{4}-\d{2}-\d{2})\n"
        r"meeting_type: (?P<meeting_type>scheduled|unscheduled)\n"
        r"document_id: (?P<document_id>[^\n]+)\n\n"
        rf"(?P<body>.*?)(?=\n{re.escape(END)})\n{re.escape(END)}",
        re.DOTALL,
    )
    matches = list(pattern.finditer(text))
    reconstructed = "\n\n".join(m.group(0) for m in matches) + "\n"
    if len(matches) != EXPECTED["documents"] or reconstructed != text:
        raise ValueError("Structural marker parse was not lossless.")
    records = []
    for match, meta in zip(matches, manifest["documents"]):
        record = {"document_id": match["document_id"], "statement_date": match["date"], "meeting_type": match["meeting_type"], "split": meta["split"], "body": match["body"]}
        expected_fields = (meta["document_id"], meta["statement_date"], meta["meeting_type"])
        if (record["document_id"], record["statement_date"], record["meeting_type"]) != expected_fields:
            raise ValueError(f"Document order/header mismatch at {meta['document_id']}")
        if len(record["body"]) != meta["normalized_characters"] or sha256_text(record["body"]) != meta["normalized_sha256"]:
            raise ValueError(f"Body integrity mismatch for {record['document_id']}")
        expected_split = "train" if record["statement_date"] <= "2024-12-31" else "validation"
        if record["split"] != expected_split or (record["split"] == "validation") != record["statement_date"].startswith("2025-"):
            raise ValueError(f"Split/date mismatch for {record['document_id']}")
        records.append(record)
    if [r["statement_date"] for r in records] != sorted(r["statement_date"] for r in records):
        raise ValueError("Documents are not in chronological order.")
    return records, manifest, checks

CORPUS_DIR = resolve_corpus_dir()
documents, manifest, integrity_checks = load_verified_corpus(CORPUS_DIR)
print("Corpus integrity PASS:", all(integrity_checks.values()), "|", CORPUS_DIR)


## 3. Chronological train/validation split

Documents through 2024-12-31 form training; 2025 documents form chronological out-of-time language-model validation. This is not a financial backtest. Document identity is disjoint and every later window is constructed within one statement body.


In [ ]:
train_docs = [d for d in documents if d["split"] == "train"]
val_docs = [d for d in documents if d["split"] == "validation"]
assert {d["document_id"] for d in train_docs}.isdisjoint(d["document_id"] for d in val_docs)
assert len(train_docs) == 82 and len(val_docs) == 8
assert sum(len(d["body"]) for d in train_docs) == 237840
assert sum(len(d["body"]) for d in val_docs) == 17474
assert Counter(d["meeting_type"] for d in documents) == {"scheduled": 87, "unscheduled": 3}

def available_windows(records, block_size): return sum(max(0, len(d["body"]) - block_size) for d in records)
summary_rows = []
for name, records in (("train", train_docs), ("validation", val_docs)):
    summary_rows.append({"split": name, "documents": len(records), "date_range": f"{records[0]['statement_date']} to {records[-1]['statement_date']}", "body_characters": sum(len(d['body']) for d in records), "windows@128": available_windows(records, 128)})
for row in summary_rows: print(row)


## 4. Character tokenizer

The deterministic vocabulary is the sorted set of characters in training statement bodies only, preserving case. Metadata and structural markers never enter model text. Validation and the fixed future prompts must be covered by the training vocabulary; vocabulary data is directly serializable in checkpoints.


In [ ]:
EVAL_PROMPTS = [
    "The Committee decided to", "Inflation has", "The labor market",
    "The target range for the federal funds rate",
    "In assessing the appropriate stance of monetary policy",
]
training_bodies = tuple(d["body"] for d in train_docs)
training_body_chars = sorted(set().union(*(set(body) for body in training_bodies)))
chars = training_body_chars.copy()
stoi = {ch: i for i, ch in enumerate(chars)}; itos = {i: ch for ch, i in stoi.items()}
VOCAB_SIZE = len(chars)

def encode(text: str) -> list[int]:
    unknown = sorted(set(text) - set(stoi))
    if unknown: raise ValueError(f"Unknown characters: {unknown!r}")
    return [stoi[ch] for ch in text]

def decode(ids: Sequence[int]) -> str:
    try: return "".join(itos[int(i)] for i in ids)
    except KeyError as exc: raise ValueError(f"Token ID outside vocabulary: {exc.args[0]}") from exc

serialized_headers = tuple(
    line
    for d in documents
    for line in (
        START, f'date: {d["statement_date"]}', f'meeting_type: {d["meeting_type"]}',
        f'document_id: {d["document_id"]}', END,
    )
)
model_documents = training_bodies
tokenizer_inputs = training_bodies
marker_exclusion = (
    chars == sorted(set().union(*(set(body) for body in training_bodies)))
    and model_documents == tuple(d["body"] for d in train_docs)
    and tokenizer_inputs == tuple(d["body"] for d in train_docs)
    and all(START not in body and END not in body for body in model_documents)
    and all(header not in body.splitlines() for body in model_documents for header in serialized_headers)
    and all(not body.startswith((START, "date: ", "meeting_type: ", "document_id: ")) for body in model_documents)
)
record_check("body-only vocabulary exactness", chars == training_body_chars)
record_check("structural marker/header exclusion", marker_exclusion)
record_check("training vocabulary size", VOCAB_SIZE == 73)
record_check("tokenizer train round-trip", all(decode(encode(d["body"])) == d["body"] for d in train_docs))
record_check("validation character coverage", not (set().union(*(set(d["body"]) for d in val_docs)) - set(chars)))
record_check("prompt character coverage", all(decode(encode(prompt)) == prompt for prompt in EVAL_PROMPTS))
VOCABULARY_DATA = {"chars": chars, "stoi": stoi}
print({"vocabulary": VOCAB_SIZE, "body_only_exact": "PASS", "marker_header_exclusion": "PASS", "train_round_trip": "PASS", "validation_coverage": "PASS", "prompt_coverage": "PASS"})


## 5. Scaled dot-product attention

For $Q,K,V\in\mathbb{R}^{B\times H\times T\times d_h}$, scores have shape $[B,H,T,T]$:

$$A=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_h}}+M\right),\qquad Y=AV.$$

A boolean mask uses `True` for allowed keys. Masking occurs before softmax; every query row must retain at least one key. Returned weights are pre-dropout, so their rows sum to one and masked probabilities are zero. Dropout is applied only to weights used for the value mixture during training.


In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None, dropout_p=0.0, training=False, return_weights=False):
    if q.ndim != 4 or k.ndim != 4 or v.ndim != 4: raise ValueError("Q, K, V must each have shape [B,H,T,d_head].")
    if q.shape != k.shape or q.shape != v.shape: raise ValueError(f"Q, K, V shapes must match; got {q.shape}, {k.shape}, {v.shape}.")
    if not q.is_floating_point() or q.dtype != k.dtype or q.dtype != v.dtype: raise TypeError("Q, K, V must share a floating dtype.")
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))  # [B,H,T,T]
    if mask is not None:
        if mask.dtype != torch.bool: raise TypeError("Attention mask must be boolean (True=allowed).")
        try: allowed = torch.broadcast_to(mask.to(scores.device), scores.shape)
        except RuntimeError as exc: raise ValueError(f"Mask shape {tuple(mask.shape)} is not broadcastable to scores {tuple(scores.shape)}.") from exc
        if not allowed.any(dim=-1).all(): raise ValueError("Every attention row must allow at least one key.")
        scores = scores.masked_fill(~allowed, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    mixed_weights = F.dropout(weights, p=dropout_p, training=training)
    output = mixed_weights @ v  # [B,H,T,d_head]
    return (output, weights) if return_weights else output


## 6. Numerical attention verification

This hand-checkable example specifies an independent NumPy reference and fixed expected numbers, rather than invoking the PyTorch implementation twice. It shows Q, K, V, scaled scores, unmasked weights, and causal weights.


In [ ]:
q_small = torch.tensor([[[[1., 0.], [0., 1.]]]])
k_small = torch.tensor([[[[1., 0.], [0., 1.]]]])
v_small = torch.tensor([[[[1., 2.], [3., 4.]]]])
scores_small = q_small @ k_small.transpose(-2, -1) / math.sqrt(2)
_, unmasked = scaled_dot_product_attention(q_small, k_small, v_small, return_weights=True)
causal2 = torch.tril(torch.ones(2, 2, dtype=torch.bool))[None, None]
out_masked, masked = scaled_dot_product_attention(q_small, k_small, v_small, mask=causal2, return_weights=True)

np_scores = np.array([[1., 0.], [0., 1.]]) / np.sqrt(2.0)
np_shift = np_scores - np_scores.max(axis=-1, keepdims=True)
np_unmasked = np.exp(np_shift) / np.exp(np_shift).sum(axis=-1, keepdims=True)
expected_unmasked = np.array([[0.66976155, 0.33023845], [0.33023845, 0.66976155]])
expected_masked = np.array([[1.0, 0.0], [0.33023845, 0.66976155]])
expected_output = expected_masked @ np.array([[1., 2.], [3., 4.]])
record_check("numerical attention shapes", scores_small.shape == (1,1,2,2) and out_masked.shape == (1,1,2,2))
record_check("independent numerical attention reference", np.allclose(np_unmasked, expected_unmasked, atol=1e-7) and np.allclose(unmasked.numpy()[0,0], expected_unmasked, atol=1e-7) and np.allclose(out_masked.numpy()[0,0], expected_output, atol=1e-7))
record_check("attention rows normalize", torch.allclose(masked.sum(-1), torch.ones_like(masked.sum(-1))))
record_check("future attention is zero", np.allclose(masked.numpy()[0,0], expected_masked, atol=1e-7) and torch.count_nonzero(masked[..., 0, 1]) == 0)
print("Q=", q_small, "\nK=", k_small, "\nV=", v_small)
print("scaled scores=", scores_small, "\nunmasked=", unmasked, "\ncausal=", masked)
print("Numerical attention PASS")


## 7. Multi-head causal self-attention

A single projection creates Q/K/V, then reshapes `[B,T,3D]` into three tensors `[B,H,T,d_head]`. Heads are reassembled to `[B,T,D]` before the output projection. Attention dropout affects weights; residual dropout affects the projected output.


In [ ]:
class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout, max_seq_len):
        super().__init__()
        if d_model % num_heads: raise ValueError("d_model must be divisible by num_heads.")
        self.num_heads, self.head_dim = num_heads, d_model // num_heads
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout, self.resid_dropout = dropout, nn.Dropout(dropout)
        self.register_buffer("causal_mask", torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))[None,None])
    def forward(self, x, return_attention=False):
        if x.ndim != 3: raise ValueError("Attention input must have shape [B,T,D].")
        B,T,D = x.shape
        if T > self.causal_mask.size(-1): raise ValueError(f"Context length {T} exceeds configured maximum {self.causal_mask.size(-1)}.")
        qkv = self.qkv(x).view(B,T,3,self.num_heads,self.head_dim).permute(2,0,3,1,4)
        q,k,v = qkv.unbind(0)  # each [B,H,T,d_head]
        y, weights = scaled_dot_product_attention(q,k,v,self.causal_mask[:,:,:T,:T],self.attn_dropout,self.training,True)
        y = y.transpose(1,2).contiguous().view(B,T,D)
        y = self.resid_dropout(self.out_proj(y))
        return (y, weights) if return_attention else y


## 8. Positional encoding

Self-attention alone is permutation-equivariant, so fixed sinusoidal positions are registered as a non-parameter buffer. The table stops at the configured maximum: neither training nor inference invents an unseen position inside one forward pass.


In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len):
        super().__init__()
        if d_model <= 0 or max_seq_len <= 0: raise ValueError("d_model and max_seq_len must be positive.")
        position = torch.arange(max_seq_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0,d_model,2,dtype=torch.float32) * (-math.log(10000.0)/d_model))
        pe = torch.zeros(max_seq_len,d_model); pe[:,0::2] = torch.sin(position*div)
        pe[:,1::2] = torch.cos(position*div[:pe[:,1::2].shape[1]])
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        if x.ndim != 3: raise ValueError("Positional input must have shape [B,T,D].")
        if x.size(1) > self.pe.size(1): raise ValueError(f"Context length {x.size(1)} exceeds positional maximum {self.pe.size(1)}.")
        if x.size(2) != self.pe.size(2): raise ValueError("Embedding dimension does not match positional encoding.")
        return x + self.pe[:,:x.size(1)].to(dtype=x.dtype)


## 9. Pre-LN Transformer block with GELU

The block is `x = x + attention(LN1(x))`, followed by `x = x + FFN(LN2(x))`. Its feed-forward path is `d_model → d_ff → d_model` with GELU and dropout.


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout, max_seq_len):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = MultiHeadCausalSelfAttention(d_model,num_heads,dropout,max_seq_len)
        self.ffn = nn.Sequential(nn.Linear(d_model,d_ff), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_ff,d_model), nn.Dropout(dropout))
    def forward(self, x, return_attention=False):
        attn_result = self.attn(self.ln1(x), return_attention=return_attention)
        if return_attention: attn_out, weights = attn_result
        else: attn_out = attn_result
        x = x + attn_out; x = x + self.ffn(self.ln2(x))
        return (x, weights) if return_attention else x


## 10. Finance MiniGPT

The model uses token embeddings, fixed positions, a configurable decoder stack, final LayerNorm, and a bias-free tied language-model head. Forward accepts `[B,T]` integer token IDs and returns logits `[B,T,V]`, plus optional next-character cross-entropy. Invalid rank, dtype, IDs, and context overflow are rejected instead of silently truncated.


In [ ]:
class FinanceMiniGPT(nn.Module):
    def __init__(self, vocab_size, architecture):
        super().__init__(); self.vocab_size = vocab_size; self.architecture = dict(architecture)
        D, L, H, FF, BS, P = (architecture[k] for k in ("d_model","num_layers","num_heads","d_ff","block_size","dropout"))
        self.token_embedding = nn.Embedding(vocab_size,D)
        self.position = SinusoidalPositionalEncoding(D,BS); self.dropout = nn.Dropout(P)
        self.blocks = nn.ModuleList([TransformerBlock(D,H,FF,P,BS) for _ in range(L)])
        self.ln_f = nn.LayerNorm(D); self.lm_head = nn.Linear(D,vocab_size,bias=False)
        self.apply(self._init_weights); self.lm_head.weight = self.token_embedding.weight
        self.architecture["parameter_count"] = sum(p.numel() for p in self.parameters())
    @staticmethod
    def _init_weights(module):
        if isinstance(module,(nn.Linear,nn.Embedding)): nn.init.normal_(module.weight,0.0,0.02)
        if isinstance(module,nn.Linear) and module.bias is not None: nn.init.zeros_(module.bias)
        if isinstance(module,nn.LayerNorm): nn.init.ones_(module.weight); nn.init.zeros_(module.bias)
    def forward(self, idx, targets=None, return_attentions=False):
        if idx.ndim != 2: raise ValueError("Token input must have rank 2: [B,T].")
        if idx.dtype not in (torch.int8,torch.int16,torch.int32,torch.int64,torch.uint8): raise TypeError("Token input must use an integer dtype.")
        if idx.numel() and (idx.min().item() < 0 or idx.max().item() >= self.vocab_size): raise ValueError(f"Token IDs must be in [0,{self.vocab_size-1}].")
        if idx.size(1) > self.architecture["block_size"]: raise ValueError(f"Context length {idx.size(1)} exceeds block_size={self.architecture['block_size']}; crop only in generation.")
        if targets is not None and (targets.shape != idx.shape or targets.dtype != torch.long): raise ValueError("Targets must be torch.long with the same [B,T] shape as input.")
        x = self.dropout(self.position(self.token_embedding(idx)))
        maps=[]
        for block in self.blocks:
            if return_attentions: x,w = block(x,True); maps.append(w)
            else: x = block(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None if targets is None else F.cross_entropy(logits.reshape(-1,self.vocab_size),targets.reshape(-1))
        return {"logits": logits, "loss": loss, "attentions": maps if return_attentions else None}


## 11. Architecture and gradient verification

These bounded tests instantiate a tiny model, not the candidate baseline. They cover shapes, finite loss and gradients, an optimizer update, causal non-leakage, overflow handling, and shared weight storage.


In [ ]:
TINY_ARCH = {"block_size": 16,"d_model": 32,"num_heads": 4,"num_layers": 2,"d_ff": 64,"dropout": 0.0}
seed_everything(0); tiny_model = FinanceMiniGPT(VOCAB_SIZE,TINY_ARCH).to(DEVICE)
sample = torch.tensor([encode(train_docs[0]["body"][:12])],dtype=torch.long,device=DEVICE)
targets = torch.tensor([encode(train_docs[0]["body"][1:13])],dtype=torch.long,device=DEVICE)
mha = MultiHeadCausalSelfAttention(32,4,0.0,16).to(DEVICE)
record_check("multi-head output shape", mha(torch.randn(2,7,32,device=DEVICE)).shape == (2,7,32))
block = TransformerBlock(32,4,64,0.0,16).to(DEVICE)
record_check("block shape invariance", block(torch.randn(2,7,32,device=DEVICE)).shape == (2,7,32))
result=tiny_model(sample,targets)
record_check("logits shape", result["logits"].shape==(1,12,VOCAB_SIZE))
record_check("finite scalar loss", result["loss"].ndim==0 and torch.isfinite(result["loss"]))
tiny_model.zero_grad(set_to_none=True); result["loss"].backward()
grads=[p.grad for p in tiny_model.parameters() if p.grad is not None]
record_check("finite nonzero gradients", grads and all(torch.isfinite(g).all() for g in grads) and any(torch.count_nonzero(g)>0 for g in grads))
optimizer=torch.optim.AdamW(tiny_model.parameters(),lr=1e-3); before=[p.detach().clone() for p in tiny_model.parameters()]
optimizer.step()
record_check("optimizer step changes parameters", any(not torch.equal(a,b) for a,b in zip(before,tiny_model.parameters())))
tiny_model.eval(); a=sample.clone(); b=sample.clone(); b[0,-1]=(b[0,-1]+1)%VOCAB_SIZE
with torch.no_grad(): la=tiny_model(a)["logits"]; lb=tiny_model(b)["logits"]
record_check("causal non-leakage", torch.equal(la[:,:-1],lb[:,:-1]))
overflow_raised = False
try: tiny_model(torch.zeros(1,17,dtype=torch.long,device=DEVICE))
except ValueError as exc: overflow_raised = "block_size" in str(exc)
record_check("context overflow rejection", overflow_raised)
record_check("weight tying storage", tiny_model.lm_head.weight.data_ptr()==tiny_model.token_embedding.weight.data_ptr())
record_check("serialized architecture parameter count", tiny_model.architecture["parameter_count"] == sum(p.numel() for p in tiny_model.parameters()))

seed_everything(7)
init_model = FinanceMiniGPT(VOCAB_SIZE,TINY_ARCH)
initialized_weights = [m.weight.detach().flatten() for m in init_model.modules() if isinstance(m, (nn.Linear, nn.Embedding))]
all_initialized = torch.cat(initialized_weights)
init_statistics_ok = abs(float(all_initialized.mean())) < 0.01 and 0.012 < float(all_initialized.std()) < 0.028
zero_linear_biases = all(torch.count_nonzero(m.bias) == 0 for m in init_model.modules() if isinstance(m, nn.Linear) and m.bias is not None)
record_check("GPT-style initialization statistics", init_statistics_ok, f"mean={float(all_initialized.mean()):.5f}, std={float(all_initialized.std()):.5f}")
record_check("zero linear biases", zero_linear_biases)

dropout_arch = {**TINY_ARCH, "dropout": 0.5}
seed_everything(11); dropout_model = FinanceMiniGPT(VOCAB_SIZE,dropout_arch).to(DEVICE)
dropout_model.train()
with torch.no_grad(): train_a=dropout_model(sample)["logits"]; train_b=dropout_model(sample)["logits"]
dropout_model.eval()
with torch.no_grad(): eval_a=dropout_model(sample)["logits"]; eval_b=dropout_model(sample)["logits"]
record_check("dropout train stochasticity", not torch.equal(train_a,train_b))
record_check("dropout eval determinism", torch.equal(eval_a,eval_b))
print("Architecture, initialization, gradient, and dropout tests PASS")


## 12. Document-aware datasets and batches

The index maps each example to `(document, start)` without duplicating window tensors. Each item yields shifted `x` and `y` of length `block_size`; no window crosses a statement boundary. A supplied generator supports deterministic random batches, while dataset order supports exhaustive validation.


In [ ]:
class DocumentWindowDataset(Dataset):
    def __init__(self, records, block_size, stoi):
        if block_size < 1: raise ValueError("block_size must be positive.")
        self.block_size=block_size; self.encoded=[]; self.index=[]
        for doc_i,d in enumerate(records):
            ids=torch.tensor([stoi[ch] for ch in d["body"]],dtype=torch.long); self.encoded.append(ids)
            self.index.extend((doc_i,start) for start in range(max(0,len(ids)-block_size)))
        if not self.index: raise ValueError(f"No document has at least block_size+1={block_size+1} characters.")
    def __len__(self): return len(self.index)
    def __getitem__(self,i):
        doc_i,start=self.index[i]; ids=self.encoded[doc_i]; return ids[start:start+self.block_size],ids[start+1:start+self.block_size+1]
    def sample_batch(self,batch_size,generator,device):
        choices=torch.randint(len(self),(batch_size,),generator=generator)
        pairs=[self[int(i)] for i in choices]; return torch.stack([p[0] for p in pairs]).to(device),torch.stack([p[1] for p in pairs]).to(device)

for bs in (64,128,256):
    train_n=available_windows(train_docs,bs); val_n=available_windows(val_docs,bs)
    assert train_n==len(DocumentWindowDataset(train_docs,bs,stoi)) and val_n==len(DocumentWindowDataset(val_docs,bs,stoi))
    print({"block_size":bs,"train_windows":train_n,"validation_windows":val_n})


## 13. Training protocol

The interface below provides AdamW decayed/non-decayed groups, gradient clipping, warmup plus cosine decay, deterministic sampling, finite checks, validation hooks, token/time accounting, and metrics records. The Phase 3A guard makes a canonical call impossible. Phase 3B must approve batch size, learning rate, and token budget first.


In [ ]:
def adamw_parameter_groups(model, weight_decay):
    decay=[]; no_decay=[]; membership={}
    for name,p in model.named_parameters():
        if not p.requires_grad: continue
        use_decay = p.ndim >= 2 and name != "token_embedding.weight"
        (decay if use_decay else no_decay).append(p)
        membership[name] = "decay" if use_decay else "no_decay"
    decay_ids, no_decay_ids = {id(p) for p in decay}, {id(p) for p in no_decay}
    trainable_ids = {id(p) for p in model.parameters() if p.requires_grad}
    if decay_ids & no_decay_ids: raise AssertionError("AdamW parameter groups overlap.")
    if decay_ids | no_decay_ids != trainable_ids: raise AssertionError("AdamW parameter groups do not cover every trainable parameter exactly once.")
    return [
        {"params":decay,"weight_decay":weight_decay,"group_name":"decay"},
        {"params":no_decay,"weight_decay":0.0,"group_name":"no_decay"},
    ], membership

def configure_adamw(model, lr, weight_decay, betas=(0.9,0.95)):
    groups, membership = adamw_parameter_groups(model, weight_decay)
    optimizer = torch.optim.AdamW(groups,lr=lr,betas=betas)
    return optimizer, membership

def cosine_lr(step,total_steps,warmup_steps,max_lr,min_lr):
    """Warmup step 0 uses max_lr/warmup_steps; with zero warmup, step 0 uses max_lr. The last optimization step is min_lr."""
    if total_steps < 1: raise ValueError("total_steps must be positive.")
    if not 0 <= step < total_steps: raise ValueError("step must be in [0, total_steps).")
    if not 0 <= warmup_steps < total_steps: raise ValueError("warmup_steps must be in [0, total_steps).")
    if not (math.isfinite(max_lr) and math.isfinite(min_lr) and 0 <= min_lr <= max_lr): raise ValueError("Learning rates must be finite with 0 <= min_lr <= max_lr.")
    if warmup_steps > 0 and step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    if total_steps == 1: return min_lr
    decay_start = warmup_steps
    denominator = total_steps - 1 - decay_start
    if denominator <= 0: return min_lr
    progress = (step - decay_start) / denominator
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

@torch.no_grad()
def evaluate_exhaustive(model,dataset,batch_size,device):
    prior=model.training; model.eval(); weighted=0.0; tokens=0
    try:
        for start in range(0,len(dataset),batch_size):
            pairs=[dataset[i] for i in range(start,min(start+batch_size,len(dataset)))]
            x=torch.stack([p[0] for p in pairs]).to(device); y=torch.stack([p[1] for p in pairs]).to(device)
            loss=model(x,y)["loss"]; weighted+=float(loss)*y.numel(); tokens+=y.numel()
        return weighted/tokens
    finally: model.train(prior)

def enter_training_mode(model, allowed):
    if not allowed: raise RuntimeError("Canonical training is disabled in Phase 3A; obtain Phase 3B approval and change the explicit guard.")
    model.train()

def train_model(model,train_ds,val_ds,training,checkpoint_context,device):
    enter_training_mode(model, CONFIG["allow_canonical_training"])
    opt,_=configure_adamw(model,training["lr"],training["weight_decay"],tuple(training["betas"])); gen=torch.Generator().manual_seed(training["seed"])
    metrics=[]; best_loss=float("inf"); best_step=None; tokens=0; started=time.perf_counter()
    for step in range(training["steps"]):
        lr=cosine_lr(step,training["steps"],training["warmup_steps"],training["lr"],training["min_lr"])
        for group in opt.param_groups: group["lr"]=lr
        x,y=train_ds.sample_batch(training["batch_size"],gen,device); opt.zero_grad(set_to_none=True)
        loss=model(x,y)["loss"]
        if not torch.isfinite(loss): raise FloatingPointError(f"Non-finite loss at step {step}")
        loss.backward(); grads=[p.grad for p in model.parameters() if p.grad is not None]
        if not grads or not all(torch.isfinite(g).all() for g in grads): raise FloatingPointError(f"Non-finite/missing gradients at step {step}")
        grad_norm=torch.nn.utils.clip_grad_norm_(model.parameters(),training["grad_clip"]); opt.step(); tokens+=y.numel()
        if step%training["validate_every"]==0 or step==training["steps"]-1:
            val_loss=evaluate_exhaustive(model,val_ds,training["eval_batch_size"],device)
            row={"step":step,"train_loss":float(loss),"validation_loss":val_loss,"lr":lr,"tokens_processed":tokens,"elapsed_seconds":time.perf_counter()-started}; metrics.append(row)
            if val_loss<best_loss: best_loss,best_step=val_loss,step; save_checkpoint(training["best_path"],"best",model,opt,None,training,checkpoint_context,step,best_step,best_loss,metrics)
    save_checkpoint(training["final_path"],"final",model,opt,None,training,checkpoint_context,training["steps"]-1,best_step,best_loss,metrics)
    return metrics

optimizer_test_model = FinanceMiniGPT(VOCAB_SIZE,TINY_ARCH)
test_weight_decay = 0.1
test_optimizer, membership = configure_adamw(optimizer_test_model,1e-3,test_weight_decay)
named = dict(optimizer_test_model.named_parameters())
decay_ids={id(p) for p in test_optimizer.param_groups[0]["params"]}; no_decay_ids={id(p) for p in test_optimizer.param_groups[1]["params"]}
all_ids={id(p) for p in optimizer_test_model.parameters() if p.requires_grad}
record_check("AdamW groups disjoint", decay_ids.isdisjoint(no_decay_ids))
record_check("AdamW full parameter coverage", decay_ids | no_decay_ids == all_ids)
record_check("AdamW group weight decay values", test_optimizer.param_groups[0]["weight_decay"] == test_weight_decay and test_optimizer.param_groups[1]["weight_decay"] == 0.0)
record_check("AdamW LayerNorm and bias no-decay", all(group == "no_decay" for name,group in membership.items() if "ln" in name or name.endswith("bias")))
record_check("AdamW embedding and tied head no-decay", membership["token_embedding.weight"] == "no_decay" and optimizer_test_model.lm_head.weight.data_ptr() == optimizer_test_model.token_embedding.weight.data_ptr() and id(optimizer_test_model.token_embedding.weight) in no_decay_ids)
record_check("AdamW matrix weights decay", all(group == "decay" for name,group in membership.items() if named[name].ndim >= 2 and name != "token_embedding.weight"))

for warmup in (0,3):
    values=[cosine_lr(step,10,warmup,1e-3,1e-4) for step in range(10)]
    record_check(f"cosine schedule finite nonnegative warmup={warmup}", all(math.isfinite(v) and v >= 0 for v in values))
    expected_start=1e-3 if warmup == 0 else 1e-3/warmup
    record_check(f"cosine schedule start warmup={warmup}", math.isclose(values[0],expected_start,rel_tol=0,abs_tol=1e-15))
    if warmup:
        record_check("cosine warmup boundary", math.isclose(values[warmup-1],1e-3,rel_tol=0,abs_tol=1e-15))
        record_check("cosine decay boundary", math.isclose(values[warmup],1e-3,rel_tol=0,abs_tol=1e-15))
    record_check(f"cosine schedule final warmup={warmup}", math.isclose(values[-1],1e-4,rel_tol=0,abs_tol=1e-15))

class GuardProbeDataset:
    sampled=False
    def sample_batch(self,*args,**kwargs): self.sampled=True; raise AssertionError("batch sampling occurred behind disabled guard")

guard_model=FinanceMiniGPT(VOCAB_SIZE,TINY_ARCH); guard_model.eval(); guard_ds=GuardProbeDataset()
guard_path=Path("/tmp/phase3a_guard_must_not_exist.pt")
guard_training={"lr":1e-3,"weight_decay":0.1,"betas":[0.9,0.95],"seed":0,"steps":1,"warmup_steps":0,"min_lr":1e-4,"batch_size":1,"grad_clip":1.0,"validate_every":1,"eval_batch_size":1,"best_path":guard_path,"final_path":guard_path}
guard_raised=False
try: train_model(guard_model,guard_ds,guard_ds,guard_training,{},torch.device("cpu"))
except RuntimeError as exc: guard_raised="disabled in Phase 3A" in str(exc)
record_check("canonical guard raises", guard_raised)
record_check("canonical guard precedes side effects", not guard_model.training and not guard_ds.sampled and not guard_path.exists())
enter_training_mode(guard_model, True)
record_check("training mode entered only after guard passes", guard_model.training)
guard_model.eval()
print("AdamW grouping, cosine schedule, and canonical guard tests PASS")


## 14. Validation and checkpoint policy

Validation is deterministic and exhaustive over body-local windows. Lower validation cross-entropy selects the best checkpoint; the final checkpoint records terminal state separately. Checkpoints carry reconstruction inputs rather than depending on notebook globals, and loading is `map_location` safe.


In [ ]:
def environment_metadata():
    return {"python":platform.python_version(),"torch":torch.__version__,"platform":platform.platform(),"device":str(DEVICE)}

def save_checkpoint(path,kind,model,optimizer,scheduler,training_config,context,step,best_step,best_loss,metrics):
    payload={"artifact_version":"finance-minigpt-checkpoint-v1","checkpoint_kind":kind,"model_state":model.state_dict(),"architecture_config":model.architecture,"training_config":training_config,"vocabulary":context["vocabulary"],"corpus_id":context["corpus_id"],"corpus_sha256":context["corpus_sha256"],"split_metadata":context["split_metadata"],"seed":context["seed"],"current_step":step,"best_step":best_step,"best_loss":best_loss,"optimizer_state":None if optimizer is None else optimizer.state_dict(),"scheduler_state":None if scheduler is None else scheduler.state_dict(),"metrics_history":metrics,"environment":environment_metadata()}
    torch.save(payload,Path(path))

def load_checkpoint(path,map_location="cpu"):
    payload=torch.load(Path(path),map_location=map_location,weights_only=False)
    if payload.get("artifact_version")!="finance-minigpt-checkpoint-v1": raise ValueError("Unsupported checkpoint schema.")
    model=FinanceMiniGPT(len(payload["vocabulary"]["chars"]),payload["architecture_config"]); model.load_state_dict(payload["model_state"])
    return model,payload


## 15. Controlled generation

Generation validates prompts, crops context only for inference, uses greedy or temperature/top-k decoding, and restores the caller's train/eval mode. Stochastic sampling uses an explicit local generator, isolating it from global RNG state. Both token IDs and decoded text are returned.


In [ ]:
@torch.no_grad()
def generate(model,prompt,stoi,itos,max_new_tokens,temperature=0.0,top_k=None,seed=None,device=None):
    unknown=sorted(set(prompt)-set(stoi))
    if unknown: raise ValueError(f"Prompt contains unknown characters: {unknown!r}")
    if temperature<0: raise ValueError("temperature must be nonnegative.")
    if top_k is not None and top_k<1: raise ValueError("top_k must be positive.")
    device=device or next(model.parameters()).device; ids=torch.tensor([[stoi[ch] for ch in prompt]],dtype=torch.long,device=device)
    prior=model.training; model.eval(); gen=None
    if temperature>0: gen=torch.Generator(device=device.type).manual_seed(CONFIG["seed"] if seed is None else seed)
    try:
        for _ in range(max_new_tokens):
            context=ids[:,-model.architecture["block_size"]:]
            logits=model(context)["logits"][:,-1]
            if temperature==0: next_id=logits.argmax(-1,keepdim=True)
            else:
                logits=logits/temperature
                if top_k is not None:
                    threshold=torch.topk(logits,min(top_k,logits.size(-1))).values[:,-1,None]; logits=logits.masked_fill(logits<threshold,float("-inf"))
                next_id=torch.multinomial(F.softmax(logits,-1),1,generator=gen)
            ids=torch.cat((ids,next_id),dim=1)
    finally: model.train(prior)
    flat=ids[0].tolist(); return {"token_ids":flat,"text":"".join(itos[i] for i in flat)}

tiny_model.train(); greedy1=generate(tiny_model,EVAL_PROMPTS[0],stoi,itos,8,device=DEVICE)
train_mode_restored=tiny_model.training
greedy2=generate(tiny_model,EVAL_PROMPTS[0],stoi,itos,8,device=DEVICE)
stoch1=generate(tiny_model,EVAL_PROMPTS[0],stoi,itos,8,0.8,10,123,DEVICE)
stoch2=generate(tiny_model,EVAL_PROMPTS[0],stoi,itos,8,0.8,10,123,DEVICE)
tiny_model.eval(); generate(tiny_model,EVAL_PROMPTS[0],stoi,itos,1,device=DEVICE)
record_check("generation mode restoration", train_mode_restored and not tiny_model.training)
record_check("deterministic greedy generation", greedy1==greedy2)
record_check("seeded stochastic generation", stoch1==stoch2)
print("Generation mode restoration and reproducibility PASS (samples intentionally not displayed)")


## 16. Evaluation scorecard

These independent diagnostics are not combined into a synthetic score. Lower character KL and repeated-trigram rate are generally preferable; higher in-corpus-word fraction and distinct-n are generally preferable; numerical-format validity is a fraction where higher means more matched numeric tokens satisfy the declared format. Empty-input behavior is defined below. Distributional resemblance is not semantic coherence or factual correctness.


In [ ]:
def character_kl(sample,reference,alphabet,eps=1e-12):
    """Lower is closer. Empty sample -> 0; empty reference with nonempty sample -> inf. Character frequency ignores nothing; smoothing limits zeros. Not semantic."""
    if not sample:return 0.0
    if not reference:return float("inf")
    p=np.array([sample.count(c) for c in alphabet],float); q=np.array([reference.count(c) for c in alphabet],float); p=(p+eps)/(p.sum()+eps*len(p)); q=(q+eps)/(q.sum()+eps*len(q)); return float(np.sum(p*np.log(p/q)))
def in_corpus_word_fraction(sample,reference_words):
    """Higher is more lexically familiar. No sample words -> 0. Case-sensitive whitespace words; not grammaticality."""
    words=sample.split(); return 0.0 if not words else sum(w in reference_words for w in words)/len(words)
def distinct_n(text,n):
    """Higher means more unique word n-grams. Fewer than n words -> 0. Sensitive to length/tokenization; not quality."""
    words=text.split(); grams=list(zip(*(words[i:] for i in range(n)))); return 0.0 if not grams else len(set(grams))/len(grams)
def repeated_word_trigram_rate(text):
    """Lower means fewer repeated word trigrams. No trigrams -> 0. Exact-match repetition only; may penalize legitimate phrases."""
    words=text.split(); grams=list(zip(words,words[1:],words[2:])); return 0.0 if not grams else (len(grams)-len(set(grams)))/len(grams)
def numerical_format_validity(text):
    """Higher means regex-detected numeric tokens have allowed finance-like forms. No numeric tokens -> 1. Regex coverage is limited, not factual validation."""
    candidates=re.findall(r"\S*\d\S*",text)
    if not candidates:return 1.0
    valid=re.compile(r"^[\$+-]?(?:\d+(?:\.\d+)?|\d+-\d+/\d+|\d+/\d+)(?:%|percent|basis)?[.,;:]?$")
    return sum(bool(valid.fullmatch(x)) for x in candidates)/len(candidates)

reference="Committee inflation 2 percent Committee policy"; score_sample="Committee inflation 2 percent"
scorecard_checks = (
    character_kl("",reference,chars)==0
    and in_corpus_word_fraction("",set(reference.split()))==0
    and distinct_n("a b a",1)==2/3
    and distinct_n("a b",2)==1
    and repeated_word_trigram_rate("a b c a b c")==1/4
    and numerical_format_validity("rate 2 percent and 1/4") == 1.0
)
record_check("scorecard unit checks", scorecard_checks)
print("Scorecard unit checks PASS")


## 17. Attention visualization

This interface later plots one explicitly selected layer and head against decoded character positions. It validates bounds and uses the causal attention map returned by the model. Phase 3A does not create the final PNG.


In [ ]:
def plot_attention(model,text,stoi,itos,layer,head,device=None):
    import matplotlib.pyplot as plt
    ids=encode(text); device=device or next(model.parameters()).device
    if len(ids)>model.architecture["block_size"]: raise ValueError("Visualization text exceeds block_size.")
    prior=model.training; model.eval()
    try:
        with torch.no_grad(): maps=model(torch.tensor([ids],dtype=torch.long,device=device),return_attentions=True)["attentions"]
    finally: model.train(prior)
    if not 0<=layer<len(maps): raise IndexError("layer is out of range.")
    if not 0<=head<maps[layer].size(1): raise IndexError("head is out of range.")
    matrix=maps[layer][0,head].cpu().numpy(); labels=[f"{i}:{repr(itos[t])}" for i,t in enumerate(ids)]
    fig,ax=plt.subplots(figsize=(7,6)); image=ax.imshow(matrix,vmin=0,vmax=matrix.max(),cmap="viridis")
    ax.set(xticks=range(len(labels)),yticks=range(len(labels)),xticklabels=labels,yticklabels=labels,title=f"Causal attention — layer {layer}, head {head}",xlabel="key",ylabel="query")
    ax.tick_params(axis="x",rotation=90); fig.colorbar(image,ax=ax,label="attention probability"); fig.tight_layout(); return fig,ax


## 18. Phase 3A bounded verification

This cell performs only a temporary tiny checkpoint reconstruction/reload equivalence check and consolidates the required correctness assertions. It writes under `/tmp`; it neither trains the candidate model nor creates repository artifacts.


In [ ]:
checkpoint_context={"vocabulary":VOCABULARY_DATA,"corpus_id":manifest["corpus_id"],"corpus_sha256":manifest["corpus_sha256"],"split_metadata":summary_rows,"seed":CONFIG["seed"]}
tmp_checkpoint=Path("/tmp/finance_minigpt_phase3a_tiny_checkpoint.pt")
tiny_model.eval(); save_checkpoint(tmp_checkpoint,"phase3a-test",tiny_model,None,None,{"purpose":"bounded reload test"},checkpoint_context,0,0,float(result["loss"].detach()),[])
reloaded,payload=load_checkpoint(tmp_checkpoint,map_location=DEVICE); reloaded.to(DEVICE).eval()
with torch.no_grad(): original_logits=tiny_model(sample)["logits"]; reloaded_logits=reloaded(sample)["logits"]
record_check("checkpoint reload equivalence", torch.equal(original_logits,reloaded_logits))
record_check("checkpoint corpus and vocabulary", payload["corpus_sha256"]==EXPECTED["sha256"] and payload["vocabulary"]["chars"]==chars)
record_check("checkpoint serialized parameter count", payload["architecture_config"]["parameter_count"] == sum(p.numel() for p in reloaded.parameters()))

expected_checks = {
    "body-only vocabulary exactness", "structural marker/header exclusion", "training vocabulary size",
    "tokenizer train round-trip", "validation character coverage", "prompt character coverage",
    "numerical attention shapes", "independent numerical attention reference", "attention rows normalize", "future attention is zero",
    "multi-head output shape", "block shape invariance", "logits shape", "finite scalar loss", "finite nonzero gradients",
    "optimizer step changes parameters", "causal non-leakage", "context overflow rejection", "weight tying storage",
    "serialized architecture parameter count", "GPT-style initialization statistics", "zero linear biases",
    "dropout train stochasticity", "dropout eval determinism", "AdamW groups disjoint", "AdamW full parameter coverage",
    "AdamW group weight decay values", "AdamW LayerNorm and bias no-decay", "AdamW embedding and tied head no-decay",
    "AdamW matrix weights decay", "cosine schedule finite nonnegative warmup=0", "cosine schedule start warmup=0",
    "cosine schedule final warmup=0", "cosine schedule finite nonnegative warmup=3", "cosine schedule start warmup=3",
    "cosine warmup boundary", "cosine decay boundary", "cosine schedule final warmup=3", "canonical guard raises",
    "canonical guard precedes side effects", "training mode entered only after guard passes", "generation mode restoration",
    "deterministic greedy generation", "seeded stochastic generation", "scorecard unit checks",
    "checkpoint reload equivalence", "checkpoint corpus and vocabulary", "checkpoint serialized parameter count",
}
missing=expected_checks-TEST_RESULTS.keys(); unexpected=TEST_RESULTS.keys()-expected_checks
if missing or unexpected: raise AssertionError(f"Phase 3A summary mismatch; missing={sorted(missing)}, unexpected={sorted(unexpected)}")
failed={name:passed for name,passed in TEST_RESULTS.items() if not passed}
if failed: raise AssertionError(f"Phase 3A failures: {failed}")
print("Phase 3A correctness suite PASS:",len(TEST_RESULTS),"derived checks")
print("Temporary checkpoint reload equivalence PASS:",tmp_checkpoint)


## 19. Findings, limitations and reproducibility

**Observed in Phase 3A:** the frozen corpus, chronological split, body-only tokenizer, transparent architecture, causal behavior, gradient/update path, deterministic generation controls, scorecard units, and temporary checkpoint reconstruction pass bounded correctness checks.

**Not yet observed:** canonical training loss, validation loss, generated samples, training curves, or attention plots. Their later cells must be clearly treated as Phase 3B outputs. The explicit `allow_canonical_training=False` guard prevents a full run here.

The experiment can test character-level distributional regularities under chronological out-of-time validation. It cannot establish policy understanding, factual correctness, prediction skill, economic value, or trading performance. Results will depend on the small corpus, character tokenization, chosen compute budget, stochastic optimization, and device-level floating-point behavior.


In [ ]:
# PHASE 3B CANONICAL TRAINING / RESULTS CELL — intentionally disabled and unexecuted in Phase 3A.
assert CONFIG["allow_canonical_training"] is False
print("Canonical training guard PASS: full training did not run.")
